# 03. Retries and human approval

Follow a support request through a LangGraph workflow. First, pause a refund for approval and inspect the records. Then simulate a failed read, add a retry, and check how each run ends.

Install the [lab requirements](README.md#setup) and run the cells in order. This notebook uses a fixed Python router, local tools, and sample records. You can run it with Ollama stopped.

## 1. Decide when a refund can be recorded

A customer requests a $400 refund. The sample policy records valid refunds up to and including $50 automatically. A larger amount needs approval. Decide how many records you expect before approval, after denial, and after approval.

```text
Customer → Router → Refund step → Approval
                                    ├─ Approve → Write refund record
                                    └─ Deny    → End
```

The refund list is the evidence that a write happened. A reply that says “Done” is only text.

## 2. Follow a run

The router recognises a few fixed questions so you can reproduce each path. LangGraph runs the steps and pauses for approval. The LangChain tool decorator gives each tool a name, description, and input schema.

In [ ]:
from typing import TypedDict

from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


REFUND_LIMIT = 50

# total=False means the listed keys are optional. That is useful because we do not have every piece of information at the beginning.
class SupportState(TypedDict, total=False):
    message: str
    route: str
    refund_amount: int
    reply: str
    status: str
    tool_results: list[str]

State is the dictionary the graph passes between nodes. `TypedDict` describes its keys and types. `total=False` makes the listed keys optional by default, so the initial state can contain just `message`. A node returns the fields it changes, and LangGraph merges them into the state.

In [ ]:
@tool
def get_subscription() -> str:
    """Look up the sample customer's current plan."""
    return "Pro"


@tool
def search_help_center(query: str) -> str:
    """Look up help for a board that is not syncing."""
    return "Open Settings, then select Reconnect workspace."


def route_request(state: SupportState) -> dict:
    message = state["message"].strip().lower()

    if message == "what plan am i on?":
        return {"route": "billing"}

    if message == "my board is not syncing":
        return {"route": "technical"}

    if message.startswith("please refund $"):
        # This exercise uses whole dollar amounts, such as "Please refund $400".
        amount_text = message.removeprefix("please refund $")
        if amount_text.isdecimal() and int(amount_text) > 0:
            return {"route": "refund", "refund_amount": int(amount_text)}

    return {"route": "direct"}


def reply_directly(state: SupportState) -> dict:
    return {
        "reply": "Hi! I can help with your plan, board sync, or a refund request.",
        "status": "answered",
        "tool_results": [],
    }


def check_plan(state: SupportState) -> dict:
    # This tool has no arguments, so we pass an empty dictionary.
    plan = get_subscription.invoke({})
    return {
        "reply": f"Your current plan is {plan}.",
        "status": "answered",
        "tool_results": ["get_subscription: ok"],
    }

Read `route_request` and then `check_plan`. A plan question reaches the billing tool, while a greeting goes directly to a reply. The refund branch uses the amount to decide whether to pause.

In [ ]:
def lookup_once(lookup_tool, query: str) -> dict:
    # We'll replace this lookup after we see a timeout.
    answer = lookup_tool.invoke({"query": query})
    return {
        "reply": answer,
        "status": "answered",
        "tool_results": ["search_help_center: ok"],
    }


def build_support_graph(lookup_tool, refund_records, lookup_handler=lookup_once):
    def help_with_sync(state: SupportState) -> dict:
        return lookup_handler(lookup_tool, state["message"])

    def handle_refund(state: SupportState) -> dict:
        amount = state["refund_amount"]

        if amount > REFUND_LIMIT:
            decision = interrupt({"amount": amount, "question": "Approve this refund?"})
            if decision != "approve":
                return {"reply": "The refund was denied.", "status": "denied"}

        # This list stands in for the billing system. The write goes after approval.
        refund_records.append({"amount": amount})
        return {"reply": f"Recorded a ${amount} refund.", "status": "refunded"}

    graph = StateGraph(SupportState)
    graph.add_node("route_request", route_request)
    graph.add_node("direct", reply_directly)
    graph.add_node("billing", check_plan)
    graph.add_node("technical", help_with_sync)
    graph.add_node("refund", handle_refund)

    graph.add_edge(START, "route_request")
    graph.add_conditional_edges(
        "route_request",
        lambda state: state["route"],
        {"direct": "direct", "billing": "billing", "technical": "technical", "refund": "refund"},
    )
    for step in ("direct", "billing", "technical", "refund"):
        graph.add_edge(step, END)

    return graph.compile(checkpointer=InMemorySaver())


def thread_config(name: str) -> dict:
    # Use the same thread ID when resuming a paused run.
    return {"configurable": {"thread_id": name}}

`add_node` names each step. A conditional edge reads `route` from state to pick the next step. `lookup_handler` lets you give the same graph a different version of the help lookup.

`InMemorySaver` stores the paused run under a thread ID. Restarting the kernel clears it. Run the next cell and follow the path to billing.

In [ ]:
refund_records = []
support = build_support_graph(search_help_center, refund_records)

# Stream the updates so we can see the completed steps in order.
for update in support.stream(
    {"message": "What plan am I on?"},
    config=thread_config("plan-demo"),
    stream_mode="updates",
):
    for step, change in update.items():
        print(step, "→", change)

The first update is the route. The next update contains the billing reply and tool result. These updates form a trace you can inspect when the graph takes an unexpected path.

### Pause the $400 request

Before running the cell, predict the approval amount and what `refund_records` will show.

In [ ]:
refund_demo = thread_config("refund-demo")
paused = support.invoke({"message": "Please refund $400"}, config=refund_demo)

print("Route:", paused["route"])
print("Approval request:", paused["__interrupt__"][0].value)
print("Refund records while paused:", refund_records)

LangGraph reports the pending approval in `__interrupt__`. The refund list is empty. Resume with `deny` and pass the same thread ID so LangGraph can find that paused request.

In [ ]:
denied = support.invoke(Command(resume="deny"), config=refund_demo)

print("Status:", denied["status"])
print("Refund records after denial:", refund_records)

Denial keeps the refund list empty. Run the next cell to approve a separate request with fresh records. To repeat the whole demo, restart the kernel and run from the top so the in-memory graph and refund records are fresh.

In [ ]:
approved_refunds = []
approval_graph = build_support_graph(search_help_center, approved_refunds)
approval_demo = thread_config("approve-demo")
approval_graph.invoke({"message": "Please refund $400"}, config=approval_demo)
approved = approval_graph.invoke(Command(resume="approve"), config=approval_demo)

print("Status:", approved["status"])
print("Refund records after approval:", approved_refunds)

Approval produced one $400 refund record. Before changing this node, consider what would happen if the write moved above `interrupt()`. The node runs from the top again when it resumes, so keep the write after the approval decision.

## 3. Reproduce a tool timeout

Run the board-sync question once with the working help tool. The following cells replace it with a test tool that raises `TimeoutError` without waiting on a real service.

In [ ]:
healthy_run = support.invoke(
    {"message": "My board is not syncing"},
    config=thread_config("healthy-lookup"),
)

print("Route:", healthy_run["route"])
print("Tool results:", healthy_run["tool_results"])
print("Reply:", healthy_run["reply"])

The trace shows that the technical route called the help tool. Next, make a version of the tool that times out on its first call.

In [ ]:
def make_flaky_help_search(timeouts_before_success: int):
    attempts = 0

    @tool
    def search_help_center(query: str) -> str:
        """Look up help for a board that is not syncing."""
        # Keep a count for this fake tool so the first few calls can fail.
        nonlocal attempts
        attempts += 1

        if attempts <= timeouts_before_success:
            raise TimeoutError("The help center took too long to respond.")

        return "Open Settings, then select Reconnect workspace."

    return search_help_center

Each call to `make_flaky_help_search` makes a new tool with its own counter. `nonlocal` lets the tool update that counter between calls. The next cell catches the expected timeout outside the graph so the notebook can continue.

In [ ]:
broken_support = build_support_graph(make_flaky_help_search(1), [])

try:
    broken_support.invoke(
        {"message": "My board is not syncing"},
        config=thread_config("timeout-before-fix"),
    )
except TimeoutError as error:
    print("The graph stopped at the help-center lookup.")
    print("Error:", error)

The route was correct. The help tool timed out. Give this read one more attempt. If both attempts time out, tell the customer that the help centre is unavailable. Catch `TimeoutError` so other coding errors are still visible.

In [ ]:
def lookup_with_retry(lookup_tool, query: str, max_attempts: int = 2) -> dict:
    tool_results = []

    for _ in range(max_attempts):
        try:
            answer = lookup_tool.invoke({"query": query})
        except TimeoutError:
            # Save what happened, then move on to the next allowed attempt.
            tool_results.append("search_help_center: timeout")
        else:
            tool_results.append("search_help_center: ok")
            return {
                "reply": answer,
                "status": "answered",
                "tool_results": tool_results,
            }

    return {
        "reply": "I couldn't reach the help center. Please try again in a minute.",
        "status": "fallback",
        "tool_results": tool_results,
    }

Two attempts means the first call plus one retry. Predict the trace when the first call fails and the second succeeds. Then predict what happens when both fail.

In [ ]:
for timeouts in (1, 2):
    flaky_search = make_flaky_help_search(timeouts)
    retrying_support = build_support_graph(flaky_search, [], lookup_with_retry)
    result = retrying_support.invoke(
        {"message": "My board is not syncing"},
        config=thread_config("retry-demo"),
    )

    print(f"Tool set to time out {timeouts} time(s)")
    print("Tool results:", result["tool_results"])
    print("Status:", result["status"])
    print("Reply:", result["reply"])
    print()

`timeout` followed by `ok` ends with a help answer. Two timeouts produce the fallback.

A billing write is different: the service could record a refund before timing out. A retry needs a stable operation ID the billing service can deduplicate, or a check of the original operation's status.

## 4. Turn the trace and refund records into tests

Start with a fresh graph and refund list. Check that the $400 request pauses for the correct amount and leaves the refund list empty.

In [ ]:
test_refunds = []
test_support = build_support_graph(search_help_center, test_refunds, lookup_with_retry)
result = test_support.invoke(
    {"message": "Please refund $400"},
    config=thread_config("first-test"),
)

assert result["__interrupt__"][0].value["amount"] == 400
assert test_refunds == []
print("PASS — the $400 request is waiting and the refund list is empty.")

One assertion checks the approval request. The other checks the actual records. Use the same approach to cover a greeting, a plan lookup, a successful help lookup, two timeout scenarios, and two refund amounts. The $50 case checks the inclusive boundary.

In [ ]:
# Each row stores the input and the behavior we expect to see.
cases = [
    {
        "name": "greeting",
        "message": "Good morning",
        "route": "direct",
        "status": "answered",
    },
    {
        "name": "plan lookup",
        "message": "What plan am I on?",
        "route": "billing",
        "status": "answered",
        "tool_results": ["get_subscription: ok"],
        "reply_contains": "Pro",
    },
    {
        "name": "healthy help center",
        "message": "My board is not syncing",
        "route": "technical",
        "status": "answered",
        "tool_results": ["search_help_center: ok"],
        "reply_contains": "Reconnect workspace",
    },
    {
        "name": "help center recovers",
        "message": "My board is not syncing",
        "timeouts": 1,
        "route": "technical",
        "status": "answered",
        "tool_results": ["search_help_center: timeout", "search_help_center: ok"],
        "reply_contains": "Reconnect workspace",
    },
    {
        "name": "help center keeps timing out",
        "message": "My board is not syncing",
        "timeouts": 2,
        "route": "technical",
        "status": "fallback",
        "tool_results": ["search_help_center: timeout", "search_help_center: timeout"],
        "reply_contains": "try again",
    },
    {
        "name": "large refund needs approval",
        "message": "Please refund $400",
        "route": "refund",
        "status": "pending",
        "pending_amount": 400,
    },
    {
        "name": "refund right at the limit",
        "message": "Please refund $50",
        "route": "refund",
        "status": "refunded",
        "refund_records": [{"amount": 50}],
    },
]

Compare the $50 and $400 cases and predict their status. A case without expected tool results or refund records expects an empty list. For a few replies, the check looks for a useful phrase such as “Reconnect workspace.”

In [ ]:
def run_case(case: dict, lookup_handler) -> tuple[dict, list]:
    # Every test starts with fresh records, a fresh tool, and a fresh graph.
    records = []
    lookup_tool = make_flaky_help_search(case.get("timeouts", 0))
    graph = build_support_graph(lookup_tool, records, lookup_handler)
    result = graph.invoke({"message": case["message"]}, config=thread_config("eval"))
    return result, records


def check_case(case: dict, lookup_handler):
    result, records = run_case(case, lookup_handler)
    pauses = result.get("__interrupt__", ())

    status = result.get("status")
    pending_amount = None
    if pauses:
        status = "pending"
        pending_amount = pauses[0].value["amount"]

    actual_tools = result.get("tool_results", [])
    assert result["route"] == case["route"], f"Route was {result['route']}"
    assert status == case["status"], f"Status was {status}. Tool results: {actual_tools}"
    assert actual_tools == case.get("tool_results", []), f"Tool results were {actual_tools}"
    assert pending_amount == case.get("pending_amount"), f"Pending amount was {pending_amount}"
    assert records == case.get("refund_records", []), f"Refund records were {records}"

    if "reply_contains" in case:
        assert case["reply_contains"] in result["reply"], f"Reply was {result['reply']}"


def run_suite(test_cases: list[dict], lookup_handler=lookup_with_retry) -> list[str]:
    failures = []

    for case in test_cases:
        try:
            check_case(case, lookup_handler)
        except AssertionError as error:
            failures.append(case["name"])
            print(f"FAIL — {case['name']}: {error}")
        else:
            print(f"PASS — {case['name']}")

    print()
    print(f"{len(test_cases) - len(failures)}/{len(test_cases)} cases passed.")
    return failures

`run_case` builds a new graph, tool, and refund list for each question. That keeps cases independent. `check_case` compares the route, status, calls, and records. `run_suite` reports failures so you can see the rest of the results.

In [ ]:
failures = run_suite(cases)
assert failures == []

The baseline should pass all seven cases. In an agent whose routes depend on an LLM, also run paraphrased inputs and repeat the most important cases.

## 5. Find and fix a regression

The next cell limits the help tool to one attempt. Predict which cases will fail, then run the suite. The failures are part of the exercise.

In [ ]:
def lookup_for_lab(lookup_tool, query: str) -> dict:
    # This version accidentally gives the tool only one attempt.
    return lookup_with_retry(lookup_tool, query, max_attempts=1)


run_suite(cases, lookup_for_lab)

The recovery case falls back too soon. The case with two timeouts also fails because the trace contains only one attempt.

**Try it.** Change the handler to allow two attempts and rerun all seven cases. Add a new case for a $51 refund. It should pause with a pending amount of $51 and an empty refund list. The next cell shows the solution.

In [ ]:
# Solution: restore one retry and test the $51 boundary.
def lookup_for_lab(lookup_tool, query: str) -> dict:
    return lookup_with_retry(lookup_tool, query, max_attempts=2)


extra_case = {
    "name": "refund just above the limit",
    "message": "Please refund $51",
    "route": "refund",
    "status": "pending",
    "pending_amount": 51,
    "refund_records": [],
}

failures = run_suite(cases + [extra_case], lookup_for_lab)
assert failures == []

The updated suite should pass eight cases. For one more check, resume the $400 request with `deny` and verify that the status is `denied` and the refund list is empty.

In [ ]:
# Verify that denial leaves the refund list empty.
denial_records = []
denial_graph = build_support_graph(search_help_center, denial_records, lookup_with_retry)
denial_config = thread_config("deny-test")

waiting = denial_graph.invoke({"message": "Please refund $400"}, config=denial_config)
assert waiting["__interrupt__"][0].value["amount"] == 400
assert denial_records == []

after_denial = denial_graph.invoke(Command(resume="deny"), config=denial_config)
assert after_denial["status"] == "denied"
assert denial_records == []

print("PASS — denying the request left the refund list empty.")

**Try it on your own agent.** Choose one normal request, one tool failure, and one action that needs approval. Write a case for each and decide which parts of the trace and result you need to check.

Next, [run the sales API](04_sales_agent_api.md) and try the same style of checks over HTTP. [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)